# 10장 (실습 2): 로그로 Shadow AI를 찾고 처리 방향 정하기

> 책 『AI 보안, 클라우드에서 시작한다』 10장 [실습] 2번을 노트북에서 단계별로 실행하는 버전입니다. 웹 접속, Agent의 Tool 호출, 업무 DB 접근 기록을 실습 1의 등록 파일과 대조해 허용 목록 밖의 AI를 찾고, 처리 방향을 정하고, 등록 후보를 정책 검사로 이어 봅니다. 소요 시간은 30~40분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 웹·SaaS, Agent·MCP, 업무 DB 세 관측 지점의 기록을 **허용 목록(등록 파일)과 대조**해 Shadow AI를 찾습니다.
2. 알려진 AI 서비스 카탈로그만으로는 **찾지 못하는** Shadow AI가 있다는 것을 확인하고, 호출 형식·계정 구분·식별자 같은 단서를 함께 씁니다.
3. 발견 항목의 처리 방향(정식 등록, 대체, 차단, 기한 있는 예외)을 정하고, 등록 후보를 실습 1의 **정책 검사로 잇는** 순환을 확인합니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | 예정 |
| 1 | 기록 살펴보기 | 예정 |
| 2 | 대조 실행 | 예정 |
| 3 | 처리 방향 정하기 | 예정 |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

## 실습 시나리오

가상 조직의 일주일(2026년 10월 13~19일) 기록을 모았습니다. 보안팀은 직원들이 승인된 기업용 생성형 AI를 쓰고 있다고 생각하지만, 실제로 어떤 AI가 어디서 쓰이는지는 확인한 적이 없습니다.

```text
 logs/proxy.csv (웹 접속) ───────┐
 logs/mcp-gateway.jsonl (Tool 호출) ┼─▶ shadow_ai.py ◀─ 등록 파일(허용 목록), AI 카탈로그, CMDB
 logs/db-access.csv (DB 접근) ────┘        │
                                          ├─▶ out/shadow-report.md (발견·처리 제안)
                                          └─▶ out/candidates.yaml (등록 후보) ─▶ 실습 1의 check.sh
```

실습의 조직, 사람, 로그는 모두 가상입니다.

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Python 3와 PyYAML(Ubuntu 24.04에는 보통 설치되어 있음, 없으면 `sudo apt install -y python3-yaml`), 4단계의 정책 검사용 Docker가 필요합니다. API Key는 필요 없습니다.
>
> **2. Jupyter 준비하기 (WSL 터미널에서 한 번만)**<br>
> - 2장 노트북([ch02_lab1_container_isolation.ipynb](../ch02-container-security/ch02_lab1_container_isolation.ipynb))의 실습 전 확인사항과 같습니다. `~/.venv-lab`에 jupyterlab을 설치한 뒤 이 폴더에서 `~/.venv-lab/bin/jupyter lab`을 실행합니다.
> - 노트북은 **반드시 `ch10-policy-as-code-shadow-ai` 폴더에서 열어야** 합니다. 실습 파일을 이 폴더 기준 경로로 읽습니다.
>
> **3. 기준 버전**<br>
> - 아래 출력은 2026년 10월 23일 Ubuntu 24.04(WSL2), Python 3.12, PyYAML 6.0.1, Docker 29.1.3, OPA 1.0.0에서 실행한 결과입니다. 시각, 번호, ID 같은 값은 실행할 때마다 다릅니다.

> 📌 (참고) 이 노트북은 실습 1과 따로 할 수 있도록 실습 파일을 처음부터 다시 복사합니다. 실습 1에서 고친 AI-006과 추가한 Egress 규칙이 없으므로, 4단계의 위반 건수는 README(실습 1을 이어서 한 경우)와 다릅니다.

## 0. 실습 준비와 실습 파일

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 기록 살펴보기 | 예정 |
| 2 | 대조 실행 | 예정 |
| 3 | 처리 방향 정하기 | 예정 |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [1]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "python3": "python3 --version",
    "docker": "docker --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

print("PyYAML:", sh("python3 -c 'import yaml; print(yaml.__version__)'") or "❌ 없음")

python3    ✅ Python 3.12.3
docker     ✅ Docker version 29.1.3, build 29.1.3-0ubuntu3~24.04.2
PyYAML: 6.0.1


`[표 실습10-4]` 실습 2의 구성

| 파일 | 역할 |
|---|---|
| `shadow-ai/logs/proxy.csv` | SASE·프록시의 웹 접속 기록 611건. 사용자, 부서, 접속한 주소와 경로, 계정 구분(기업·개인), 올린 데이터 크기 |
| `shadow-ai/logs/mcp-gateway.jsonl` | MCP Gateway를 지난 Agent의 Tool 호출 기록 105건 |
| `shadow-ai/logs/db-access.csv` | 업무 DB 접근 기록 193건. 접속한 앱, 계정, 출발지 |
| `shadow-ai/catalog/ai-domains.csv` | 알려진 AI 서비스의 도메인과 분류 14개 |
| `shadow-ai/cmdb/apps.csv` | CMDB에 등록된 일반 업무 시스템(9-1) |
| `shadow-ai/shadow_ai.py` | 기록을 카탈로그·등록 파일·CMDB와 대조해 발견 목록, 처리 제안, 등록 후보 초안을 만듦 |

실습 파일을 `~/ch10-lab`으로 복사합니다.

In [2]:
import csv

LAB10 = Path.home() / "ch10-lab"
SHADOW, POLICY = LAB10 / "shadow-ai", LAB10 / "policy-as-code"
sh(f"rm -rf {LAB10} && cp -r '{LAB}' {LAB10} && rm -f {LAB10}/*.ipynb && chmod +x {POLICY}/check.sh")
REG = POLICY / "registry" / "ai-assets.yaml"


def shadow():
    """shadow_ai.py를 실행해 출력을 돌려줍니다. 터미널의 python3 shadow_ai.py와 같습니다."""
    return subprocess.run(["python3", "shadow_ai.py"], cwd=SHADOW, capture_output=True, text=True).stdout


def check():
    r = subprocess.run(["./check.sh"], cwd=POLICY, capture_output=True, text=True, env=dict(os.environ, TODAY="2026-10-20"))
    return r.stdout


print(sorted(p.name for p in SHADOW.rglob("*") if p.is_file()))

['ai-domains.csv', 'apps.csv', 'db-access.csv', 'mcp-gateway.jsonl', 'proxy.csv', 'shadow_ai.py']


## 1. 기록 살펴보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 기록 살펴보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 대조 실행 | 예정 |
| 3 | 처리 방향 정하기 | 예정 |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [3]:
%%bash
cd ~/ch10-lab/shadow-ai
head -4 logs/proxy.csv
head -2 logs/mcp-gateway.jsonl
head -3 logs/db-access.csv

time,user,dept,host,path,tenant,bytes_out
2026-10-13T08:26:36,u0190,설계팀,chatgpt.com,/backend-a

pi/conversation,기업,6569
2026-10-13T08:40:56,u0174,개발팀,github.com,/,-,3389
2026-10-13T08:45

:05,u0188,설계팀,chat.deepseek.com,/api/v0/file/upload_file,-,700978


{"time": "2026-10-13T08:15:36", "agent_id": "AGT-mkt-sns", "dept": "마케팅팀", "mcp_host": "mcp.

social-poster.example.com", "tool": "post_message", "result": "허용"}
{"time": "2026-10-13T08:46:1

5", "agent_id": "AGT-purchase", "dept": "구매팀", "mcp_host": "mcp-erp.corp.example", "tool": "cr

eate_purchase_order", "result": "허용"}
time,db,account,client_app,src_ip
2026-10-13T08:15:23,CRM,

svc_sales_ai,sales-summary,10.30.5.31
2026-10-13T08:15:36,ERP,svc_erp,erp-core,10.10.1.11


`tenant`는 같은 서비스를 기업 계정으로 썼는지 개인 계정으로 썼는지를 나타냅니다. SASE·CASB는 기업 계정의 로그인 도메인이나 테넌트 정보를 보고 이를 구분합니다.

웹 접속 기록에서 많이 접속한 주소를 세어 봅니다. 이 목록만 봐서는 무엇이 AI이고, 무엇이 승인된 것인지 알 수 없습니다.

In [4]:
from collections import Counter

proxy = list(csv.DictReader(open(SHADOW / "logs" / "proxy.csv", encoding="utf-8")))
print(f"웹 접속 기록 {len(proxy)}건")
table(["주소(host)", "접속 수", "부서 수"],
      [(h, n, len({r["dept"] for r in proxy if r["host"] == h})) for h, n in Counter(r["host"] for r in proxy).most_common(12)])

웹 접속 기록 611건


| 주소(host) | 접속 수 | 부서 수 |
|---|---|---|
| chatgpt.com | 153 | 8 |
| copilot.microsoft.com | 60 | 8 |
| mail.corp.example | 40 | 8 |
| github.com | 38 | 8 |
| claude.ai | 38 | 1 |
| docs.python.org | 37 | 8 |
| portal.corp.example | 36 | 8 |
| erp.corp.example | 35 | 8 |
| stackoverflow.com | 33 | 7 |
| www.google.com | 28 | 8 |
| www.naver.com | 28 | 8 |
| m365.cloud.microsoft | 25 | 8 |

## 2. 대조 실행

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 기록 살펴보기 | ✅ 완료 |
| 2 | 대조 실행 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 처리 방향 정하기 | 예정 |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [5]:
print(shadow())

Shadow AI 발견 결과 (프록시 611건 중 AI 336건, 등록 자산 8개와 대조)

[웹·SaaS]
  승인                     ChatGPT (AI-001)                         8개 부서               93명 140건 업로드 0.7MB
  승인                     Microsoft Copilot (AI-002)               8개 부서               52명 60건 업로드 0.2MB
  승인                     Microsoft 365 Copilot (AI-002)           8개 부서               24명 25건 업로드 87KB
  승인                     DeepL (AI-008)                           영업팀                 12명 12건 업로드 81KB
  미등록 AI                DeepSeek                                 설계팀                 1명 6건 업로드 5.1MB  ◀ 우선
  승인 서비스·개인 계정    ChatGPT (AI-001)                         재무팀                 2명 9건 업로드 2.3MB  ◀ 우선
  미등록 AI                Claude                                   개발팀                 4명 38건 업로드 0.5MB  ◀ 보통
  미등록 AI                Gamma                                    마케팅팀               3명 15건 업로드 0.2MB  ◀ 보통
  카탈로그 밖 AI API       ec2-3-35-120-17.ap-northeast-2.compute.a 연구소                 2명 22건 업로드 0.2MB  

세 가지 기록에서 서로 다른 종류의 Shadow AI가 드러납니다.

`[표 실습10-5]` 발견 유형과 찾은 방법

| 발견 유형 | 이 실습의 예 | 무엇으로 찾았나 |
|---|---|---|
| 미등록 AI | DeepSeek, Claude, Gamma, Hugging Face | 접속 주소를 AI 서비스 카탈로그와 대조 |
| 승인 서비스·개인 계정 | 재무팀의 ChatGPT 개인 계정 | 같은 서비스라도 계정 구분(`tenant`)이 개인 |
| 카탈로그 밖 AI API | 연구소가 클라우드 서버에 직접 띄운 모델 | 주소는 모르지만 호출 경로가 LLM API 형식(`/v1/chat/completions`) |
| 승인 Agent의 미승인 Tool | 코딩 Agent의 웹 가져오기 MCP Server | MCP Gateway 기록을 MCP Server 허용 목록과 대조 |
| 미등록 Agent | 마케팅팀의 SNS 게시 Agent | 등록 파일에 없는 Agent ID |
| 미등록 현업 앱 | ERP에 공용 관리자 계정(`erp_admin`)으로 붙은 재고 대시보드 | DB 접근 기록의 앱을 등록 파일·CMDB와 대조 |

### 2.1. 카탈로그만 썼다면

카탈로그(알려진 AI 도메인)와의 대조만으로는 무엇을 놓쳤을까요? 카탈로그에 없는 주소 가운데 **호출 경로가 LLM API 모양**인 기록을 직접 찾아봅니다.

In [6]:
import re

catalog = {r["domain"] for r in csv.DictReader(open(SHADOW / "catalog" / "ai-domains.csv", encoding="utf-8"))}
LLM_API = re.compile(r"/v1/(chat/)?completions|/v1/messages|/api/(generate|chat)$")
unknown_ai = [r for r in proxy if not any(r["host"] == d or r["host"].endswith("." + d) for d in catalog)
              and LLM_API.search(r["path"])]
print(f"카탈로그 도메인 {len(catalog)}개 / 카탈로그 밖인데 LLM API 경로인 접속 {len(unknown_ai)}건")
table(["주소", "경로", "부서", "건수"],
      [(*k, n) for k, n in Counter((r["host"], r["path"], r["dept"]) for r in unknown_ai).most_common()])

카탈로그 도메인 14개 / 카탈로그 밖인데 LLM API 경로인 접속 22건


| 주소 | 경로 | 부서 | 건수 |
|---|---|---|---|
| ec2-3-35-120-17.ap-northeast-2.compute.amazonaws.com:8000 | /v1/chat/completions | 연구소 | 22 |

카탈로그만으로는 직접 띄운 모델을 찾을 수 없었습니다. 알려진 서비스 목록과의 대조는 출발점일 뿐이고, 호출 형식, 계정 구분, Agent와 앱의 식별자처럼 여러 단서를 함께 봐야 합니다(9-2).

### 2.2. 승인된 서비스, 개인 계정

같은 ChatGPT라도 계정 구분에 따라 다르게 봅니다. 부서별로 기업·개인 계정 사용과 업로드 크기를 셉니다.

In [7]:
gpt = [r for r in proxy if r["host"] in ("chatgpt.com", "chat.openai.com")]
rows = []
for (dept, tenant), n in sorted(Counter((r["dept"], r["tenant"]) for r in gpt).items()):
    up = sum(int(r["bytes_out"]) for r in gpt if r["dept"] == dept and r["tenant"] == tenant)
    rows.append((dept, tenant, n, f"{up / 1e6:.2f}MB", "◀ 개인 계정" if tenant == "개인" else ""))
table(["부서", "계정", "접속", "업로드", ""], rows)

| 부서 | 계정 | 접속 | 업로드 |  |
|---|---|---|---|---|
| 개발팀 | 기업 | 22 | 0.11MB |  |
| 고객지원팀 | 기업 | 14 | 0.06MB |  |
| 마케팅팀 | 기업 | 12 | 0.06MB |  |
| 생산관리팀 | 기업 | 24 | 0.12MB |  |
| 설계팀 | 기업 | 16 | 0.08MB |  |
| 연구소 | 기업 | 6 | 0.04MB |  |
| 영업팀 | 개인 | 4 | 0.02MB | ◀ 개인 계정 |
| 영업팀 | 기업 | 34 | 0.16MB |  |
| 재무팀 | 개인 | 9 | 2.29MB | ◀ 개인 계정 |
| 재무팀 | 기업 | 12 | 0.06MB |  |

## 3. 처리 방향 정하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 기록 살펴보기 | ✅ 완료 |
| 2 | 대조 실행 | ✅ 완료 |
| 3 | 처리 방향 정하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

보고서에는 발견 항목마다 처리 제안이 있고, 마지막 칸은 담당자가 결정을 적도록 비워 두었습니다.

In [8]:
display(Markdown((SHADOW / "out" / "shadow-report.md").read_text()))

# Shadow AI 발견과 처리

| 우선 | 출처 | 상태 | 대상 | 부서 | 사용 | 처리 제안 | 처리 결정 (등록/대체/폐기·차단/예외+기한) |
|---|---|---|---|---|---|---|---|
| 우선 | 웹·SaaS | 미등록 AI | DeepSeek | 설계팀 | 1명 6건, 업로드 5.1MB | 올린 내용부터 확인(유출 여부), 그다음 승인된 AI-001로 대체 | |
| 우선 | 웹·SaaS | 승인 서비스·개인 계정 | ChatGPT (AI-001) | 재무팀 | 2명 9건, 업로드 2.3MB | 대체: 기업 계정으로 전환 안내, 개인 계정 업로드는 DLP로 차단(10-3). 올린 내용 확인 | |
| 보통 | 웹·SaaS | 미등록 AI | Claude | 개발팀 | 4명 38건, 업로드 0.5MB | 대체: 같은 용도의 승인된 AI-001 안내 | |
| 보통 | 웹·SaaS | 미등록 AI | Gamma | 마케팅팀 | 3명 15건, 업로드 0.2MB | 등록 검토: 업무에 필요하면 등록 후보로 | |
| 우선 | 웹·SaaS | 카탈로그 밖 AI API | ec2-3-35-120-17.ap-northeast-2.compute.amazonaws.com:8000 | 연구소 | 2명 22건, 업로드 0.2MB | 확인: 보안성 검토 없이 운영되는 AI 시스템. 담당자를 찾아 등록 후보로(9-2) | |
| 보통 | 웹·SaaS | 승인 서비스·개인 계정 | ChatGPT (AI-001) | 영업팀 | 1명 4건, 업로드 15KB | 대체: 기업 계정으로 전환 안내, 개인 계정 업로드는 DLP로 차단(10-3) | |
| 보통 | 웹·SaaS | 미등록 AI | Hugging Face | 연구소 | 3명 5건, 업로드 3KB | 등록 검토: 내려받은 모델의 출처·서명 확인(7-3) | |
| 우선 | Agent·MCP | 승인 Agent·미승인 MCP | AGT-coding → mcp-webfetch.example.net | 개발팀 | 1명 9건 | 차단: Tool 허용 목록 위반(실습 1의 [Tool] 위반과 같은 문제) | |
| 우선 | Agent·MCP | 미등록 Agent·미승인 MCP | AGT-mkt-sns → mcp.social-poster.example.com | 마케팅팀 | 1명 14건 | 차단 후 등록 검토: MCP Gateway에서 차단, 담당 부서 확인 | |
| 우선 | 업무 DB | 미등록 현업 앱 | inventory-dash | 확인 필요 10.20.7.88 | 1명 18건 | 등록 의무: 소유자 지정, 전용 계정·최소 권한(표 10-3), 공용 관리자 계정 회수 | |


처리는 10-3의 네 갈래(정식 등록, 대체, 차단, 기한 있는 예외) 중 하나로 정합니다.

`[표 실습10-6]` 처리 결정의 예

| 발견 | 처리 | 이유 |
|---|---|---|
| DeepSeek (설계팀, 5.1MB 업로드) | 차단 + 올린 내용 확인, 승인된 AI-001로 대체 | 설계 자료가 올라갔을 수 있으므로 유출 여부부터 확인 |
| ChatGPT 개인 계정 (재무팀) | 대체: 기업 계정 전환, 개인 계정 업로드는 DLP로 차단 | 같은 서비스라도 개인 계정은 기업 계약의 데이터 보호를 받지 못함 |
| Claude (개발팀) | 업무에 필요하면 정식 등록 검토, 아니면 AI-001로 대체 | 쓰는 이유를 먼저 듣는다. 반복되면 승인 절차가 너무 느리다는 신호 |
| Gamma (마케팅팀) | 정식 등록 검토 | 승인된 대안이 없는 용도 |
| 연구소 자체 모델 API | 등록 후보: 보안성 검토 후 등록 | 보안성 검토 없이 운영되는 AI 시스템 |
| 코딩 Agent의 웹 가져오기 MCP | 차단, 필요하면 MCP Server 등록 절차로 | 실습 1의 [Tool] 위반과 같은 문제를 로그에서 확인 |
| 마케팅팀 SNS Agent | MCP Gateway에서 차단 후 등록 검토 | 외부 게시는 되돌릴 수 없는 행동 |
| 재고 대시보드 | 등록 의무, 공용 관리자 계정 회수와 전용 계정 발급 | 표 10-3의 "연결 시 등록 의무" |

막기만 하면 Shadow AI는 더 깊이 숨습니다. 차단과 함께 승인된 대안이나 등록 경로를 알려 주는 것이 처리의 기본입니다.

## 4. 등록 후보를 등록하고 정책 검사로 잇기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 기록 살펴보기 | ✅ 완료 |
| 2 | 대조 실행 | ✅ 완료 |
| 3 | 처리 방향 정하기 | ✅ 완료 |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

### 4.1. 등록 후보 초안

`shadow_ai.py`는 등록 후보마다 등록 파일 형식의 초안을 만듭니다. 재고 대시보드(NEW-5)의 초안을 봅니다.

In [9]:
print((SHADOW / "out" / "candidates.yaml").read_text())

# 등록 후보 초안. 담당 부서와 함께 빈칸을 채운 뒤 등록 파일(assets:)에 옮기고 실습 1의 check.sh로 검사합니다.
- id: NEW-1
  name: Gamma (마케팅팀)
  type: 문서·발표 생성
  status: 운영
  owner_dept: 마케팅팀
  accountable: ''
  declared_grade: ''
  data: {}
  controls: {}
  domains: [gamma.app]
- id: NEW-2
  name: 연구소 자체 모델 API
  type: AI 서비스
  status: 운영
  owner_dept: 연구소
  accountable: ''
  declared_grade: ''
  data: {}
  controls: {}
  endpoint: ec2-3-35-120-17.ap-northeast-2.compute.amazonaws.com:8000
- id: NEW-3
  name: Hugging Face (연구소)
  type: 모델 저장소
  status: 운영
  owner_dept: 연구소
  accountable: ''
  declared_grade: ''
  data: {}
  controls: {}
  domains: [huggingface.co]
- id: NEW-4
  name: AGT-mkt-sns
  type: Agent
  status: 운영
  owner_dept: 마케팅팀
  accountable: ''
  declared_grade: ''
  data: {}
  controls: {}
  agent_id: AGT-mkt-sns
- id: NEW-5
  name: inventory-dash
  type: 현업 앱
  status: 운영
  owner_dept: ''
  accountable: ''
  declared_grade: ''
  data: {}
  controls: {}
  app_id: inventory-dash



### 4.2. 빈칸 그대로 등록하기

초안을 등록 파일에 그대로 옮겨 넣으면 무엇이 비어 있는지 정책 검사가 알려 줍니다. README에서는 편집기로 `mcp_servers:` 위에 넣습니다. 노트북에서는 함수로 넣습니다.

In [10]:
AI009 = """  - id: AI-009
    name: 재고 현황 대시보드
    type: 현업 앱
    status: 운영
    owner_dept: ''
    accountable: ''
    declared_grade: ''
    app_id: inventory-dash
    data: {}
    controls: {}

"""
MARK = "# Agent가 쓸 수 있는 MCP Server"


def put_asset(block):
    """등록 파일의 MCP Server 목록 앞에 자산 항목을 넣거나, 같은 ID가 있으면 바꿉니다."""
    text = REG.read_text()
    m = re.search(r"  - id: AI-009\n.*?\n\n", text, re.S)
    text = text[:m.start()] + block + text[m.end():] if m else text.replace(MARK, block + MARK)
    REG.write_text(text)


put_asset(AI009)
print("\n".join(l for l in check().splitlines() if "AI-009" in l or "위반" in l))

  AI-009 [등급] 재고 현황 대시보드: 위험 등급이 정해지지 않았습니다 (기준 등급 '낮음')
  AI-009 [승인] 재고 현황 대시보드: '낮음' 등급은 등록 승인이 필요합니다 (현재 없음)
  AI-009 [책임] 재고 현황 대시보드: 소유 부서가 비어 있습니다
  AI-009 [책임] 재고 현황 대시보드: 최종 책임자(A)가 비어 있습니다
위반 15건, 자산 6개 → 배포·운영 승인 불가 (종료 코드 1)


### 4.3. 빈칸 채우기

담당 부서를 찾아 빈칸을 채웁니다. 대시보드는 ERP를 조회만 하므로 `internal_read`이고, 기준 등급은 중간입니다.

In [11]:
AI009 = """  - id: AI-009
    name: 재고 현황 대시보드
    type: 현업 앱
    status: 운영
    owner_dept: 생산관리팀
    accountable: 윤생산
    declared_grade: 중간
    app_id: inventory-dash
    data: {internal_read: true}
    controls: {approved_by: 부서장, last_review: "2026-10-20"}

"""
put_asset(AI009)
print("\n".join(l for l in check().splitlines() if "AI-009" in l or "위반" in l) or "AI-009 위반 없음")
print()
print("\n".join(l for l in shadow().splitlines() if "inventory" in l or "허용 목록 밖" in l))

위반 11건, 자산 5개 → 배포·운영 승인 불가 (종료 코드 1)

  승인                     inventory-dash                           생산관리팀             18건
허용 목록 밖 사용 9건 (우선 5건) — 표 10-11 '미등록 AI·Agent 발견 건수'


AI-009의 위반이 사라졌고, 다시 대조하면 재고 대시보드는 "승인"으로 바뀌어 미등록 건수가 10건에서 9건으로 줄었습니다. 이것이 10-3의 순환입니다. 찾고(실습 2), 등록하고, 등록하면 정책 검사가 따라오고(실습 1), 다시 찾습니다.

다만 정책 검사를 통과했다고 문제가 모두 풀린 것은 아닙니다. 대시보드는 여전히 공용 관리자 계정 `erp_admin`으로 ERP에 붙어 있습니다. 등록 파일에는 계정 정보가 없으므로 정책 검사는 이것을 보지 못합니다(과제 1).

In [12]:
db = list(csv.DictReader(open(SHADOW / "logs" / "db-access.csv", encoding="utf-8")))
table(["앱", "DB", "계정", "출발지", "건수"],
      [(*k, n) for k, n in Counter((r["client_app"], r["db"], r["account"], r["src_ip"]) for r in db
                                   if r["client_app"] == "inventory-dash").most_common()])

| 앱 | DB | 계정 | 출발지 | 건수 |
|---|---|---|---|---|
| inventory-dash | ERP | erp_admin | 10.20.7.88 | 18 |

`[표 실습10-7]` 실습 2와 실무 도구

| 이 실습 | 실무에서 같은 역할 (예) |
|---|---|
| `proxy.csv`와 AI 서비스 카탈로그 | CASB·SASE의 Shadow IT 발견과 AI 앱 카탈로그, 브라우저 보안 확장 |
| `mcp-gateway.jsonl` | MCP Gateway·AI Gateway의 호출 기록 |
| `db-access.csv` | DB 접근 제어·감사 솔루션의 접속 기록 |
| `shadow_ai.py`의 대조와 보고서 | SIEM의 상관 분석 규칙, AI-SPM, ITSM의 처리 티켓 |
| 미등록 건수 | 표 10-11의 "미등록 AI·Agent 발견 건수" 추이 |

## 5. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 기록 살펴보기 | ✅ 완료 |
| 2 | 대조 실행 | ✅ 완료 |
| 3 | 처리 방향 정하기 | ✅ 완료 |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | ✅ 완료 |
| 5 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 정리 | 예정 |

### 5.1. 체크포인트

- [ ] 카탈로그와의 대조만으로 찾지 못한 Shadow AI는 무엇이었고, 무엇으로 찾았나요?
- [ ] 재무팀의 ChatGPT 사용은 서비스 자체가 승인되어 있는데도 왜 문제였나요?
- [ ] 발견한 항목을 모두 차단했다면 어떤 일이 생길까요? 차단과 함께 무엇을 제공해야 할까요?
- [ ] 재고 대시보드는 정책 검사를 통과했는데도 남은 위험은 무엇이었나요? 정책 검사가 보지 못하는 것은 무엇으로 확인해야 할까요?

### 5.2. 과제 1: "관리자 계정 금지" 규칙 추가하기

정책 검사가 공용 관리자 계정을 보게 하려면, 먼저 등록 파일에 접속 계정을 적게 하고, 그 값을 검사하는 규칙을 둡니다.

1. 아래 셀은 AI-009에 `db_account: erp_admin`을 적어 넣습니다(DB 접근 기록과 같은 값).
2. `RULE`에 "`db_account`가 관리자 계정이면 [계정] 위반" 규칙을 씁니다. 관리자 계정은 이름에 `admin`이 들어가거나 `root`, `sa`인 계정으로 봅니다.
3. 실행해 AI-009가 [계정] 위반으로 걸리는지, 전용 계정(`svc_inventory_ro`)으로 바꾸면 사라지는지 확인합니다.

In [13]:
RULE = """package aigov

# TODO: db_account가 관리자 계정이면 [계정] 위반
"""
(POLICY / "policy" / "account.rego").write_text(RULE)

for account in ["erp_admin", "svc_inventory_ro"]:
    put_asset(AI009.replace("    data:", f"    db_account: {account}\n    data:"))
    lines = [l.strip() for l in check().splitlines() if "[계정]" in l]
    print(f"db_account={account:17} →", lines or "[계정] 위반 없음")

db_account=erp_admin         → [계정] 위반 없음


db_account=svc_inventory_ro  → [계정] 위반 없음


<details>
<summary>💡 힌트 보기</summary>

```python
RULE = """package aigov

deny contains msg if {
	some a in data.assets
	acct := object.get(a, "db_account", "")
	regex.match(`(?i)admin|^root$|^sa$`, acct)
	msg := sprintf("%s [계정] %s: 공용 관리자 계정(%s)으로 접속합니다. 전용 최소 권한 계정을 쓰세요", [a.id, a.name, acct])
}
"""
```

이 규칙은 **등록 파일에 적힌 값**만 봅니다. 실제로 어떤 계정으로 접속하는지는 2단계처럼 DB 접근 기록으로 확인해야 합니다. 그래서 등록 파일의 값과 실제 기록이 다르면 그 자체를 위반으로 보는 대조(등록값 ↔ 관측값)가 필요합니다. 정책 검사(실습 1)와 로그 대조(실습 2)를 함께 돌리는 이유입니다.
</details>

### 5.3. 과제 2: 처리 결정 적기 (서술)

3단계 보고서의 "처리 결정" 칸을 담당자 입장에서 채워 봅니다. 표 실습10-6에 없는 **Hugging Face(연구소)** 와 **ChatGPT 개인 계정(영업팀)** 두 건의 처리와 이유, 기한을 적어 보세요.

| 발견 | 처리 | 이유 | 기한 |
|---|---|---|---|
| Hugging Face (연구소) | | | |
| ChatGPT 개인 계정 (영업팀) | | | |

## 6. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 기록 살펴보기 | ✅ 완료 |
| 2 | 대조 실행 | ✅ 완료 |
| 3 | 처리 방향 정하기 | ✅ 완료 |
| 4 | 등록 후보를 등록하고 정책 검사로 잇기 | ✅ 완료 |
| 5 | 체크포인트와 과제 | ✅ 완료 |
| 6 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [14]:
print(sh("rm -rf ~/ch10-lab && echo '실습 폴더 삭제됨'"))

실습 폴더 삭제됨


---
> 이 책의 코드 실습은 여기까지입니다. 2장의 컨테이너 격리부터 10장의 정책 집행까지, 각 장의 통제가 "사람의 기억"이 아니라 **확인할 수 있는 기록과 자동 검사**로 이어지는지 다시 돌아보세요.